<a href="https://colab.research.google.com/github/cmlg96/spatial-data-management-with-GEE/blob/main/Lab06_cm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### This code authenticates and initializes the Google Earth Engine API and installs the earthengine-api and geemap libraries for visualizing geographical data in Jupyter Notebooks.

In [ ]:
!pip install geemap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 15.8 MB/s eta 0:00:00


In [ ]:
!pip install earthengine-api

In [ ]:
import geemap
import ee

In [ ]:
ee.Authenticate()
ee.Initialize(project='my-project')

#### Part 1: The code loads NOAA GFS 0.25° forecast data, extracts the 2m temperature layer, and visualizes it on a map. A color bar, NOAA logo, and attribution text are included for reference. The map is configured for global display.

In [ ]:
# Load NOAA GFS 0.25° Forecast Data
dataset = ee.ImageCollection("NOAA/GFS0P25")
temperature = dataset.select("temperature_2m_above_ground").first()

# Define visualization parameters
vis_params = {
    "min": -40,
    "max": 35,
    "palette": ['blue', 'purple', 'cyan', 'green', 'yellow', 'red']
}

# Create a map
Map = geemap.Map(center=[20, 0], zoom=2)

# Add the temperature layer to the map
Map.addLayer(temperature, vis_params, "NOAA GFS Temperature")

# Add a color bar
Map.add_colorbar(vis_params, label="Temperature Above Ground")

# Add NOAA logo
noaa_logo = "https://upload.wikimedia.org/wikipedia/commons/thumb/5/57/Noaa-logo-rgb-2022.svg/2048px-Noaa-logo-rgb-2022.svg.png"
Map.add_image(noaa_logo, position="bottomleft", width="100px")

# Add a text label to the map
Map.add_text("Made by Clara MR", fontsize=20, position='bottomright')

# Display the map
Map


Map(center=[20, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(child…

#### Part 2: The code loads a Landsat 7 5-year composite image and visualizes it in a 2x2 grid using different band combinations for varied analyses. Each map is linked, with labels indicating the band configurations. The maps are centered on a specified location and zoom level for detailed inspection.

In [ ]:
# Load the Landsat 7 5-year composite image
image = ee.Image('LANDSAT/LE7_TOA_5YEAR/1999_2003')

# Define different visualization parameters using valid bands
vis_params = [
    {"bands": ["B3", "B2", "B1"], "min": 0, "max": 300, "gamma": 1.3},  # Natural Color
    {"bands": ["B7", "B5", "B3"], "min": 0, "max": 300, "gamma": 1.3},  # Agriculture (Vegetation)
    {"bands": ["B4", "B3", "B2"], "min": 0, "max": 300, "gamma": 1.3},  # Color Infrared
    {"bands": ["B5", "B4", "B3"], "min": 0, "max": 300, "gamma": 1.3},  # Vegetation
]

# Create a 2x2 linked map
labels = [
    "Natural Color (B3/B2/B1)",
    "Geology and Moisture (B7/B5/B3)",
    "Color Infrared (B4/B3/B2)",
    "Vegetation (B5,B4,B3)",
]

linked_maps = geemap.linked_maps(
    rows=2,
    cols=2,
    height="400px",
    center=[38.4151, 21.2712],  # Adjust center based on your area of interest
    zoom=12,
    ee_objects=[image, image, image, image],  # Reuse the same image for each map
    vis_params=vis_params,
    labels=labels,
    label_position="topright",
)

# Display the linked maps
linked_maps



GridspecLayout(children=(Output(layout=Layout(grid_area='widget001')), Output(layout=Layout(grid_area='widget0…

#### Part 3: The process involves loading USDA NASS Cropland Data Layers from 2010 to 2022 and configuring a timeseries inspector for analysis. A region of interest is defined, and visualization parameters are applied to the cropland layers. The map is centered on the ROI, with attribution and the USDA logo included for reference.

In [ ]:
# Create a map instance.
Map = geemap.Map()

# Load USDA NASS Cropland Data Layers from 2010 to 2022.
CDL = ee.ImageCollection("USDA/NASS/CDL").filter(ee.Filter.calendarRange(2010, 2022, 'year'))
CDL_layers = CDL.aggregate_array("system:id").getInfo()
print(CDL_layers)

# Create a list of layer names for the timeseries inspector.
CDL_layer_names = ["CDL " + str(year) for year in range(2010, 2023)]

# Define a region of interest (ROI).
roi = ee.Geometry.Point([-95.7129, 37.0902]).buffer(50000)

# Configure the visualization parameters for the CDL layers.
cdl_vis = {"bands": ["cropland"]}

# Add the timeseries inspector to the map.
Map.ts_inspector(
    left_ts=CDL,
    right_ts=None,
    left_names=CDL_layer_names,
    right_names=None,
    left_vis=cdl_vis,
    right_vis=None,
)

# Center the map on the ROI.
Map.centerObject(roi, zoom=5)

# Add your name and the USDA logo to the map.
Map.add_text("Made by Clara MR", position="bottomright")
Map.add_image("https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcQY8FTi_82nr8-smxXOdBLk4s3UyUhGIBYfFw&s", position="bottomleft", width=100, height=50)

# Display the map.
Map

['USDA/NASS/CDL/2010', 'USDA/NASS/CDL/2011', 'USDA/NASS/CDL/2012', 'USDA/NASS/CDL/2013', 'USDA/NASS/CDL/2014', 'USDA/NASS/CDL/2015', 'USDA/NASS/CDL/2016', 'USDA/NASS/CDL/2017', 'USDA/NASS/CDL/2018', 'USDA/NASS/CDL/2019', 'USDA/NASS/CDL/2020', 'USDA/NASS/CDL/2021', 'USDA/NASS/CDL/2022']


Map(center=[37.09025129235146, -95.71289821683608], controls=(WidgetControl(options=['position', 'transparent_…

#### Part 4: The process loads Sentinel-2 images filtered by date, region of interest (Knoxville, TN), and cloud cover (<10%). Visualization parameters are applied for NIR, Red, and Green bands, and a time slider is added to explore monthly imagery. The map is centered on Knoxville, and attribution is included.

In [ ]:
# Create a map instance.
Map = geemap.Map()

# Define the region of interest (ROI) for Knoxville, TN.
knoxville_roi = ee.Geometry.Point([-83.9207, 35.9606]).buffer(10000)  # 10 km buffer around Knoxville

# Load Sentinel-2 images, filter by date, region, and cloud cover.
sentinel2 = (
    ee.ImageCollection("COPERNICUS/S2_SR")
    .filterBounds(knoxville_roi)
    .filterDate("2020-01-01", "2023-12-31")
    .filterMetadata("CLOUDY_PIXEL_PERCENTAGE", "less_than", 10)  # Filter images with less than 10% cloud cover
)

# Reduce the number of images in the collection by filtering one image per month.
sentinel2 = sentinel2.filter(ee.Filter.calendarRange(1, 1, 'month'))  # One image per month

# Set visualization parameters for Sentinel-2.
vis_params = {
    "min": 0,
    "max": 4000,
    "bands": ["B8", "B4", "B3"],  # NIR, Red, and Green bands for false-color visualization
    "gamma": 1.4
}

# Add the first image of the collection to the map as a reference.
first_image = sentinel2.first()
Map.addLayer(first_image, vis_params, "First image", False)

# Center the map on Knoxville, TN.
Map.setCenter(-83.9207, 35.9606, 12)

# Add a time slider to visualize Sentinel-2 images over time.
Map.add_time_slider(sentinel2, vis_params, time_interval=1, date_format='YYYY-MM-dd', opacity=0.8)

# Add attribution and your name to the map.
Map.add_text("Made by Clara Mr", position="bottomright")

# Display the map.
Map

Map(center=[35.9606, -83.9207], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchD…

#### Part 5: The process loads ESA World Cover and Landsat datasets, with both clipped to U.S. state boundaries. Visualization parameters are applied: Landsat uses infrared bands, while ESA Land Cover uses its predefined color palette. A split-map compares the two layers, and U.S. state boundaries are overlaid. The map is centered on the U.S., and attribution is included.

In [ ]:
# Load the U.S. states dataset.
states = ee.FeatureCollection("TIGER/2018/States")

# Load the ESA land cover dataset.
esa_land_cover = ee.ImageCollection("ESA/WorldCover/v100").first()

# Load the Landsat dataset (as an image, not a collection).
landsat = ee.Image("LANDSAT/LE7_TOA_5YEAR/1999_2003")

# Clip the images to the boundaries of the U.S. states.
esa_land_cover_clipped = esa_land_cover.clipToCollection(states)
landsat_clipped = landsat.clipToCollection(states)

# Define visualization parameters for ESA land cover.
esa_vis_params = {
    'min': 10,
    'max': 100,
    'palette': [
        '006400', 'FFBB22', 'FFFF4C', 'F096FF', 'FA0000', 'B4B4B4',
        'F0F0F0', '0064C8', '0096A0', '00CF75', 'FAE6A0'
    ]
}

# Define visualization parameters for Landsat.
landsat_vis_params = {
    'bands': ['B4', 'B3', 'B2'],  # Bands for Infrared Color
    'min': 0,
    'max': 300,
    'gamma': 1.3
}

# Convert Earth Engine images to tile layers.
left_layer = geemap.ee_tile_layer(landsat_clipped, landsat_vis_params, "Landsat")
right_layer = geemap.ee_tile_layer(esa_land_cover_clipped, esa_vis_params, "ESA Land Cover")

# Create a geemap map.
Map = geemap.Map()

# Add the split map.
Map.split_map(left_layer=left_layer, right_layer=right_layer)

# Add the U.S. state boundaries.
Map.addLayer(states, {}, "US States")

# Add the ESA Land Cover legend.
Map.add_legend(title='ESA Land Cover Legend', builtin_legend='ESA_WorldCover')

# Center the map on the U.S.
Map.setCenter(-98.583, 39.833, 4)  # Approximate coordinates of the U.S. center

# Add a text label to the map.
Map.add_text("Made by Clara MR", position='bottomright')

# Display the map.
Map

Map(center=[39.833, -98.583], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoo…